# Silverwing tokcache harvester (v3)

The token caches already exist inside `<USER>/silverwing-state`
(`.token-cache-*.pt` in the corpus dir). This job copies them out into the
dedicated `<USER>/silverwing-tokcache` dataset so training sessions can
fetch caches without pulling checkpoints/corpus, and so state pushes stay
small. No tokenization, no repo - just a file relay.

In [ ]:
# Cell 1: pull state, relay cache files into cacheout/
import json
import shutil
import subprocess
from pathlib import Path

USER = 'videlisndichi'
STATE = f'{USER}/silverwing-state'
TCACHE = f'{USER}/silverwing-tokcache'
WORK = Path('/kaggle/working')
STATE_DIR = WORK / 'state'
CACHEOUT = WORK / 'cacheout'

def _kaggle(*args):
    return subprocess.run(['kaggle', *args], capture_output=True, text=True)

dl = WORK / '.dl'
if dl.exists():
    shutil.rmtree(dl)
dl.mkdir(parents=True)
r = _kaggle('datasets', 'download', STATE, '-p', str(dl), '--unzip')
assert r.returncode == 0, f'download failed:\n{r.stderr[-1500:]}'
if STATE_DIR.exists():
    shutil.rmtree(STATE_DIR)
shutil.move(str(dl), STATE_DIR)
# free the fat stuff we do not need for a file relay
for sub in ('checkpoints', 'tokenizer'):
    p = STATE_DIR / sub
    if p.exists():
        shutil.rmtree(p)
CACHEOUT.mkdir(exist_ok=True)
caches = sorted((STATE_DIR / 'corpus/corpus-external').glob('.token-cache-*.pt'))
assert caches, 'no .token-cache files found in state corpus dir'
for f in caches:
    shutil.copy2(f, CACHEOUT / f.name)
    print(f'relayed {f.name} ({f.stat().st_size / 1e9:.2f} GB)')

In [ ]:
# Cell 2: push cacheout/ to the tokcache dataset
meta = {'title': 'Silverwing Tokcache', 'id': TCACHE,
        'licenses': [{'name': 'CC0-1.0'}]}
(CACHEOUT / 'dataset-metadata.json').write_text(json.dumps(meta))
exists = _kaggle('datasets', 'status', TCACHE).returncode == 0
args = ['datasets', 'version' if exists else 'create', '-p', str(CACHEOUT),
        '--dir-mode', 'zip']
if exists:
    args += ['-m', 'refresh token caches']
r = _kaggle(*args)
assert r.returncode == 0, f'push failed:\n{r.stdout[-800:]}\n{r.stderr[-800:]}'
print('tokcache dataset pushed')